In [3]:
import ipywidgets as widgets

In [4]:
import joblib
import json
import pandas as pd
from IPython.display import display, HTML

# Load the final model and metadata
try:
    model = joblib.load('airline_price_model.joblib')
    with open('model_metadata.json', 'r') as f:
        metadata = json.load(f)
    
    # Reconstruct input options from metadata
    airlines = metadata['categories']['airline']
    source_cities = metadata['categories']['source_city']
    dest_cities = metadata['categories']['destination_city']
    departure_times = metadata['categories']['departure_time']
    arrival_times = metadata['categories']['arrival_time']
    classes = metadata['categories']['class']
    stops_options = ['zero', 'one', 'two_or_more']
    
    min_dur, max_dur = metadata['duration_range']
    min_days, max_days = metadata['days_left_range']

    # Create interactive widgets
    airline_w = widgets.Dropdown(options=airlines, value='Vistara', description='Airline:')
    source_w = widgets.Dropdown(options=source_cities, value='Delhi', description='From:')
    dest_w = widgets.Dropdown(options=dest_cities, value='Mumbai', description='To:')
    dept_w = widgets.Dropdown(options=departure_times, value='Morning', description='Departure:')
    arr_w = widgets.Dropdown(options=arrival_times, value='Evening', description='Arrival:')
    class_w = widgets.Dropdown(options=classes, value='Economy', description='Class:')
    stops_w = widgets.Dropdown(options=stops_options, value='one', description='Stops:')
    
    duration_w = widgets.FloatSlider(value=10.0, min=min_dur, max=max_dur, step=0.1, description='Duration (h):')
    days_w = widgets.IntSlider(value=15, min=min_days, max=max_days, step=1, description='Days Left:')
    
    predict_btn = widgets.Button(description='Predict Price', button_style='primary')
    output_lbl = widgets.HTML(value='<h3>Enter flight details and click Predict.</h3>')

    def make_prediction(b):
        # Validate that source and destination are not identical
        if source_w.value == dest_w.value:
            output_lbl.value = '<p style="color:red;"><b>Error:</b> Source and Destination cities cannot be the same.</p>'
            return
            
        # Prepare input data matching feature_cols
        input_data = pd.DataFrame([{
            'airline': airline_w.value,
            'source_city': source_w.value,
            'departure_time': dept_w.value,
            'stops': stops_w.value,
            'arrival_time': arr_w.value,
            'destination_city': dest_w.value,
            'class': class_w.value,
            'duration': duration_w.value,
            'days_left': days_w.value
        }])
        
        # Reorder columns to match train set schema
        input_data = input_data[metadata['feature_cols']]
        
        # Predict and format the output price
        predicted_val = model.predict(input_data)[0]
        output_lbl.value = f'''
        <div style="padding:10px; background-color:#f0f8ff; border-radius:5px; border-left: 5px solid #0033A0;">
            <h4>Estimated Flight Ticket Price:</h4>
            <span style="font-size: 24px; color: #0033A0; font-weight: bold;">₹ {predicted_val:,.2f} {metadata["currency"]}</span>
        </div>
        '''

    predict_btn.on_click(make_prediction)

    # Render the input elements
    grid = widgets.GridBox(
        [airline_w, class_w, source_w, dest_w, dept_w, arr_w, stops_w, duration_w, days_w],
        layout=widgets.Layout(grid_template_columns='repeat(2, 300px)', grid_gap='10px')
    )
    display(HTML('<h3>🛩️ Airline Ticket Price Estimator</h3>'))
    display(grid)
    display(widgets.Box([predict_btn], layout=widgets.Layout(margin='15px 0px')))
    display(output_lbl)

except FileNotFoundError:
    print("Error: 'airline_price_model.joblib' or 'model_metadata.json' not found. Ensure the previous cells completed successfully and saved these files.")


/workspaces/flask-model/.venv/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OrdinalEncoder from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/workspaces/flask-model/.venv/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


AttributeError: module 'sklearn.compose._column_transformer' has no attribute '_RemainderColsList'